# Response Streaming with the Google Gemini API

Companion notebook to Anthropic's **Response Streaming** lesson, implemented with the free **Google GenAI SDK** (`gemini-3.5-flash-lite`).

### Core Concepts:
- In standard synchronous calls, generation blocks until the full message completes (10–30s on long outputs).
- Streaming delivers **real-time token chunks** over a persistent connection, slashing **Time-To-First-Token (TTFT)** down to milliseconds.
- We explore both single-request chunk streaming (`generate_content_stream`) and continuous multi-turn chat streaming (`send_message_stream`).

In [ ]:
# Install dependencies
%pip install google-genai python-dotenv

In [ ]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Initialize Gemini Client
from google import genai

client = genai.Client()
model = "gemini-3.5-flash-lite"

--- 
## Method 1: Low-Level Chunk Streaming (`generate_content_stream`)

Equivalent to Claude's `client.messages.create(stream=True)`. We iterate through chunks as they are yielded from the model.

In [ ]:
prompt = "Write a 1 sentence description of a fake database"

response_stream = client.models.generate_content_stream(
    model=model,
    contents=prompt
)

full_response = []
for chunk in response_stream:
    print(chunk.text, end="", flush=True)
    full_response.append(chunk.text)

print("\n\nTotal chunks received:", len(full_response))

--- 
## Method 2: High-Level Chat Streaming (`chat.send_message_stream`)

The official, recommended pattern for conversational apps in Google GenAI SDK.

Benefits:
1. **Automatic History Preservation**: As soon as streaming finishes, the entire response is automatically stored in `chat.get_history()`.
2. **Zero Manual Concatenation**: No need to manually assemble chunks to maintain multi-turn dialogue.

In [ ]:
# Create stateful chat session
chat = client.chats.create(model=model)

# Turn 1: Streamed response
print("--- Turn 1 (Streaming) ---")
stream_1 = chat.send_message_stream("Write a 1 sentence description of a fake database")
for chunk in stream_1:
    print(chunk.text, end="", flush=True)
print("\n")

# Turn 2: Follow-up relying on history
print("--- Turn 2 (Follow-up Streaming) ---")
stream_2 = chat.send_message_stream("What is the primary key of this database?")
for chunk in stream_2:
    print(chunk.text, end="", flush=True)
print("\n")

--- 
## Method 3: Inspecting Accumulated Conversation History

Notice how `chat.get_history()` retains the fully assembled turns without any extra boilerplate.

In [ ]:
print("Cumulative Chat History Entries:", len(chat.get_history()))
for i, message in enumerate(chat.get_history()):
    role = message.role
    text = message.parts[0].text if message.parts else ""
    print(f"[{role.upper()}]: {text}\n")